# Weights

> Keep encode-path tensors (and optional `net.*` for property serve); drop LM / AE-decoder heads.


In [ ]:
#| default_exp weight_adapters


In [ ]:
#| hide
from nbdev.showdoc import *


## The problem

`model_weights.safetensors` holds **everything** IBM trained: encoder, decoder LM, autoencoder decoder, and a property `net` head. Our MAX graph needs the **encode** path, and optionally `net.*` when serving a finetune:

- `encoder.*` (except `encoder.lang_model.*`)
- `decoder.autoencoder.encoder.*`
- `net.*` (property head; unused in embedding mode)

Everything else would confuse `load_state_dict` — so we filter.


In [ ]:
#| export
from __future__ import annotations

from collections.abc import Mapping

from max.graph.weights import WeightData, Weights


In [ ]:
#| export
# Training-only heads not used by the encode / property graphs.
_IGNORE_PREFIXES = (
    "encoder.lang_model.",
    "decoder.lang_model.",
    "decoder.autoencoder.decoder.",
)


## The adapter

MAX calls this when `WeightsFormat.safetensors` is selected. Input: Hub state dict. Output: `{name: WeightData}` for the graph.


In [ ]:
#| export
def convert_safetensor_state_dict(
    state_dict: Mapping[str, Weights],
) -> dict[str, WeightData]:
    "Filter Hub / finetune safetensors for the encode + optional ``net`` graph."
    new_state_dict: dict[str, WeightData] = {}

    for weight_name, value in state_dict.items():
        if any(weight_name.startswith(prefix) for prefix in _IGNORE_PREFIXES):
            continue
        if not (
            weight_name.startswith("encoder.")
            or weight_name.startswith("decoder.autoencoder.encoder.")
            or weight_name.startswith("net.")
        ):
            continue
        if value is None:
            continue
        new_state_dict[weight_name] = value.data()

    return new_state_dict


A tiny unit test with fake keys — no GPU required:


In [ ]:
#| hide
class _Fake:
    def __init__(self, keep=True):
        self._keep = keep
    def data(self):
        return "data"

fake = {
    "encoder.tok_emb.weight": _Fake(),
    "encoder.lang_model.lm_head.weight": _Fake(),
    "decoder.autoencoder.encoder.fc1.weight": _Fake(),
    "decoder.autoencoder.decoder.fc1.weight": _Fake(),
    "net.fc1.weight": _Fake(),
    "unrelated": _Fake(),
}
out = convert_safetensor_state_dict(fake)
assert set(out) == {
    "encoder.tok_emb.weight",
    "decoder.autoencoder.encoder.fc1.weight",
    "net.fc1.weight",
}
out


### Checkpoint

*Kept: `encoder.*` + `decoder.autoencoder.encoder.*` + `net.*`. Dropped: LM heads, AE decoder.*


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
